In [0]:
from pyspark.sql.functions import col, lag, round, row_number, count
from pyspark.sql.window import Window

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("silver_schema", "valeriimatviiv_silver", "2. Silver Schema")
dbutils.widgets.text("gold_schema", "valeriimatviiv_gold", "3. Gold Schema")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

price_silver_table = f"{catalog}.{silver_schema}.nasdaq_price_silver"
news_silver_table = f"{catalog}.{silver_schema}.finnhub_news_silver"
target_gold_table = f"{catalog}.{gold_schema}.nasdaq_news_impact_gold"

# 1. Read Silver Datasets
df_price = spark.read.table(price_silver_table)
df_news = spark.read.table(news_silver_table)

# 2. Compute Daily Stock Returns
window_symbol = Window.partitionBy("Symbol").orderBy("TradeDate")

df_price_returns = (
    df_price
    .withColumn("PrevClose", lag("Close", 1).over(window_symbol))
    .withColumn(
        "DailyReturnPct", 
        round(((col("Close") - col("PrevClose")) / col("PrevClose")) * 100.0, 4)
    )
    .filter(col("PrevClose").isNotNull())
)

# 3. Isolate QQQ Benchmark Returns
df_qqq_returns = (
    df_price_returns
    .filter(col("Symbol") == "QQQ")
    .select(
        col("TradeDate").alias("QQQ_TradeDate"),
        col("DailyReturnPct").alias("QQQ_DailyReturnPct")
    )
)

# 4. Join Company Returns with QQQ Benchmark and Calculate Relative Impact
df_company_returns = (
    df_price_returns
    .filter(col("Symbol") != "QQQ")
    .join(df_qqq_returns, col("TradeDate") == col("QQQ_TradeDate"), "left")
    .withColumn(
        "RelativeImpactPct",
        round(col("DailyReturnPct") - col("QQQ_DailyReturnPct"), 4)
    )
    .select(
        "Symbol",
        "TradeDate",
        "Close",
        "Volume",
        "DailyReturnPct",
        "QQQ_DailyReturnPct",
        "RelativeImpactPct"
    )
)

# 5. Calendar-Aware Forward Snap Join (News Date -> Next Available Trade Date)
df_company_news = df_news.filter(col("Symbol") != "GENERAL")

# Count news per company per date for context
window_daily_news = Window.partitionBy("Symbol", "NewsDate")
df_company_news = df_company_news.withColumn("NewsCountOnDay", count("ArticleId").over(window_daily_news))

# Join news to price where TradeDate >= NewsDate
df_news_price_join = df_company_news.join(
    df_company_returns,
    (df_company_news.Symbol == df_company_returns.Symbol) & 
    (df_company_returns.TradeDate >= df_company_news.NewsDate),
    "inner"
)

# Window to pick the immediate next trading day (ImpactDate)
window_snap = Window.partitionBy("ArticleId").orderBy(col("TradeDate").asc())

df_gold = (
    df_news_price_join
    .withColumn("snap_rank", row_number().over(window_snap))
    .filter(col("snap_rank") == 1)
    .select(
        col("ArticleId"),
        df_company_news["Symbol"],
        col("NewsDate"),
        col("Headline"),
        col("Summary"),
        col("TradeDate").alias("ImpactDate"),
        col("Close").alias("ImpactDayClose"),
        col("DailyReturnPct").alias("CompanyReturnPct"),
        col("QQQ_DailyReturnPct"),
        col("RelativeImpactPct"),
        col("NewsCountOnDay"),
        col("url"),
        col("source")
    )
    .orderBy(col("ImpactDate").desc(), col("RelativeImpactPct").desc())
)

# 6. Write Gold Delta Table
df_gold.write.format("delta").mode("overwrite").saveAsTable(target_gold_table)

print(f"Successfully created Gold table: '{target_gold_table}'")

In [0]:
# catalog = dbutils.widgets.get("catalog")
# gold_schema = dbutils.widgets.get("gold_schema")
# target_gold_table = f"{catalog}.{gold_schema}.nasdaq_news_impact_gold"

# df_gold_result = spark.read.table(target_gold_table)

# print(f"--- Gold Impact Table Record Count: {df_gold_result.count()} ---")
# print("--- Schema Breakdown ---")
# df_gold_result.printSchema()

# print("--- Highest Relative Impact News Articles (Top Alpha Movers) ---")
# display(
#     df_gold_result
#     .select("Symbol", "NewsDate", "ImpactDate", "Headline", "CompanyReturnPct", "QQQ_DailyReturnPct", "RelativeImpactPct")
#     .limit(10)
# )